# A Data-Driven Review Policy for Credit Card Fraud: Full Reproducible Analysis

This notebook produces every number, table and figure for the Group Report (and Figure 1 of the Project Charter), following the final project charter: an 80% review success rate and \$5 review cost as the base case, a cost-and-success sensitivity table, a clearly defined baseline rule, and an H2 test based on the confidence interval for the model-minus-rule difference. Each step explains what it does and why we chose that method.

**How to use it:** run the cells from top to bottom with **Shift + Enter**. The whole notebook takes about 2–4 minutes on Colab's free tier. The random seed is fixed at 42, so every run gives identical numbers.

| Section | What it produces |
|---|---|
| 0. Setup | Libraries, colors, settings |
| 1. Load, quality checks, clean | Data-quality audit and data profile (Report: Data and Preparation) |
| 2. Exploratory analysis | Report Figures 1, 2, B1–B4; Charter Figure 1 |
| 3. Features and leakage guard | The 34 model features |
| 4. Time-based split | Train / validation / test sets |
| 5. Two models | Table 1 (validation columns) |
| 6. Cost-based threshold | Figure 3 |
| 7. Baseline rule and test-set evaluation | Baseline-rule check, Table 1 (test columns), Table 2, Figure 4, Figure B6 |
| 8. Uncertainty and H2 test | Bootstrap CIs, CI for the model-minus-rule difference, Figure B5 |
| 9. Diagnostics | Figure 5 (calibration), Figure 6 (importance), stability, Table 3 (sensitivity to review cost and review success), robustness to data-quality issues |
| 10. Fairness audit | Table 4 |
| 11. Save and download | Zip of all figures plus results.json |

## 0. Setup

**What:** import the libraries and fix the random seed and the two cost assumptions from the project charter.

**Why these choices:**
- **pandas / NumPy** for data handling and **scikit-learn** for modeling. These are standard, free, and already installed in Colab, which keeps the analysis reproducible for anyone (Pedregosa et al., 2011).
- **`SEED = 42`** makes the bootstrap and the gradient-boosting model give identical results on every run.
- **`REVIEW_COST = 5`** is our assumed dollar cost for an analyst to review one flagged transaction.
- **`REVIEW_SUCCESS = 0.80`** is our assumed share of fraud losses that a review actually prevents when the flagged transaction is fraud. Reviewers miss some fraud, so assuming 100% would overstate the savings.
- Both are *assumptions*, not measured bank figures. As the charter promises, Section 9 re-runs the policy at review costs of \$2, \$10 and \$20 and review success rates of 60% and 100%.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             precision_recall_curve, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEED = 42
REVIEW_COST = 5.0          # assumed analyst cost (USD) per reviewed transaction
REVIEW_SUCCESS = 0.80      # assumed share of a flagged fraud's loss that review prevents (charter base case)
SENS_COSTS = [2, 5, 10, 20]          # review-cost scenarios (Section 9)
SENS_SUCCESS = [0.60, 0.80, 1.00]    # review-success scenarios (Section 9)
rng = np.random.default_rng(SEED)

FIG = Path("figures"); FIG.mkdir(exist_ok=True)
R = {}                     # every key number gets stored here and saved to results.json at the end

# One color scheme for every chart
NAVY, RED, GREEN, GOLD, GREY = "#1F5FA0", "#B31E2B", "#237A57", "#E8A33D", "#8A94A6"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11})

def save(name):
    """Save the current chart to figures/ and show it in the notebook.
    Charts carry no built-in title: the report gives each one an APA caption instead."""
    plt.tight_layout()
    plt.savefig(FIG / name, dpi=200)
    plt.show()

import sklearn; print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 1. Load the data

**Upload:** when the **Choose Files** button appears, pick `credit_card_fraud_detection_2026.csv` (about 25 MB, so it may take a minute).

> If you'd rather keep the file in Google Drive, skip the upload cell and use the commented-out Drive option instead.

In [ ]:
from google.colab import files
uploaded = files.upload()
CSV = list(uploaded.keys())[0]

# --- Option B: read from Google Drive instead ---
# from google.colab import drive
# drive.mount("/content/drive")
# CSV = "/content/drive/MyDrive/credit_card_fraud_detection_2026.csv"   # change to your path

## 1b. Data quality checks (before any cleaning)

Before changing anything, we inspect the raw file the way an auditor would. The question for every check is the same: *can we trust this column, and does it need fixing?* We load the file into `raw` and leave it untouched, so every check sees the data exactly as Kaggle delivered it. Cleaning happens afterwards, in Section 1c, on a copy.

The checks run in this order:
1. **Structure:** size, column types, a sample of rows.
2. **Missing values:** where they are, and *why* they're missing.
3. **Duplicates.**
4. **Ranges:** impossible or suspicious values.
5. **Categorical levels:** typos or inconsistent spellings.
6. **Logical consistency:** do related columns agree with each other?
7. **Target variable:** how rare fraud is, and whether that rate is stable over time.
8. **Distributions:** shape and skew of numeric features.
9. **Outliers.**
10. **Correlation:** redundant features.

In [ ]:
raw = pd.read_csv(CSV, parse_dates=["transaction_datetime"])
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)

print(f"Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns\n")
print("Column types:"); print(raw.dtypes.astype(str).value_counts().to_string(), "\n")
raw.head(3).T          # transposed so all 43 columns fit on screen

**Why look at types:** a numeric column read as text (for example, because of a stray "N/A") would silently break the model. Here the types are as expected: dates parsed as dates, IDs and categories as text, measurements as numbers.

The next cell counts unique values per column. Very few unique values means a flag or category; exactly one would mean a useless constant column; as many as there are rows means an ID.

In [ ]:
pd.DataFrame({"dtype": raw.dtypes.astype(str), "unique values": raw.nunique()})

### Missing values

Counting blanks is only half the job. The other half is asking **why** they're blank, because that decides the fix:
- *Missing at random* (a sensor failed): impute or drop.
- *Structurally missing* (the field doesn't apply to that row): fill with an explicit "None" category.

The cross-tab below tests our claim that `browser` is blank only for in-store and ATM transactions.

In [ ]:
na = raw.isna().sum()
print("Columns with missing values:")
print(na[na > 0].to_frame("missing").assign(pct=lambda t: (t.missing / len(raw) * 100).round(1)), "\n")

print("Is browser blank?  (rows) by channel:")
pd.crosstab(raw["browser"].isna().map({True: "browser blank", False: "browser filled"}),
            raw["transaction_channel"])

**Result:** only `browser` has blanks (41,709 rows, 41.7%). Every blank is ATM or In-store, and every Online or Mobile App row has a browser. The claim is verified, so filling with `"None"` is correct.

### Duplicates

We check two things: repeated transaction IDs, and rows that are identical in every column except the ID (the same purchase loaded twice under different IDs).

In [ ]:
print("Duplicate transaction IDs        :", raw.transaction_id.duplicated().sum())
print("Fully duplicated rows            :", raw.duplicated().sum())
print("Duplicated ignoring transaction ID:", raw.drop(columns="transaction_id").duplicated().sum())

### Ranges and summary statistics

`describe()` gives min, max, mean and percentiles for every numeric column. We scan for **impossible** values (negative amounts, credit scores outside 300–850, ages under 18, hours outside 0–23) and **suspicious** ones (a maximum that looks like an artificial cap).

In [ ]:
desc = raw.describe().T[["min", "25%", "50%", "75%", "max", "mean", "std"]]
desc.round(2)

In [ ]:
checks = {
    "negative transaction_amount":      (raw.transaction_amount < 0).sum(),
    "credit_score outside 300-850":     (~raw.credit_score.between(300, 850)).sum(),
    "customer_age under 18":            (raw.customer_age < 18).sum(),
    "transaction_hour outside 0-23":    (~raw.transaction_hour.between(0, 23)).sum(),
    "account_age_days < 1":             (raw.account_age_days < 1).sum(),
    "transaction_amount at max 15,000": (raw.transaction_amount == raw.transaction_amount.max()).sum(),
    "avg_amount_30d at max 2,000":      (raw.avg_transaction_amount_30d == raw.avg_transaction_amount_30d.max()).sum(),
}
pd.Series(checks, name="rows").to_frame()

**Result:** no impossible values. A few columns stop at round numbers (`transaction_amount` at \$15,000, `avg_transaction_amount_30d` at \$2,000), which suggests the data generator capped them. Only a handful of rows sit at those caps, so they have no practical effect.

### Categorical levels

We list every level of every text column. This catches spelling variants ("USA" vs "U.S.A.", "Chrome" vs "chrome"), which a model would treat as different categories.

In [ ]:
cat_cols = ["merchant_category", "merchant_country", "transaction_day_of_week", "customer_gender",
            "customer_state", "transaction_channel", "device_type", "browser", "email_domain", "fraud_type"]
for c in cat_cols:
    vc = raw[c].value_counts(dropna=False)
    print(f"{c}  ({len(vc)} levels)")
    print("   " + ", ".join(f"{k}: {v:,}" for k, v in vc.items()), "\n")

**Result:** levels are clean and consistently spelled. `customer_state` has 40 levels and `merchant_country` has 20; the rarer ones are why the one-hot encoder later uses `min_frequency=20`.

### Logical consistency

Many columns describe the same thing from different angles, so they must agree. Each check below counts the rows that **break** a rule. Zero means the rule holds.

**Why this matters:** in real data these checks catch pipeline bugs. Here they also tell us how much to trust a simulated dataset.

In [ ]:
dt = raw.transaction_datetime
fraud = raw.is_fraud == 1
rules = {
    # outcome fields
    "fraud_type is 'Legitimate' but is_fraud = 1":        ((raw.fraud_type == "Legitimate") & fraud).sum(),
    "fraud_type is a fraud type but is_fraud = 0":        ((raw.fraud_type != "Legitimate") & ~fraud).sum(),
    "loss > 0 on a legitimate transaction":               ((raw.fraud_amount_lost > 0) & ~fraud).sum(),
    "fraud with zero loss":                               ((raw.fraud_amount_lost == 0) & fraud).sum(),
    "loss larger than the transaction amount":            (raw.fraud_amount_lost > raw.transaction_amount + 0.01).sum(),
    # time fields
    "transaction_hour != hour of datetime":               (dt.dt.hour != raw.transaction_hour).sum(),
    "transaction_day_of_week != weekday of datetime":     (dt.dt.day_name() != raw.transaction_day_of_week).sum(),
    "is_weekend != weekend of datetime":                  ((dt.dt.dayofweek >= 5).astype(int) != raw.is_weekend).sum(),
    "is_weekend != weekend of day_of_week column":        (raw.transaction_day_of_week.isin(["Saturday", "Sunday"]).astype(int) != raw.is_weekend).sum(),
    # geography
    "USA merchant marked international":                  ((raw.merchant_country == "USA") & (raw.international_transaction == 1)).sum(),
    "non-USA merchant marked domestic":                   ((raw.merchant_country != "USA") & (raw.international_transaction == 0)).sum(),
    "high-risk country but not international":            ((raw.high_risk_country == 1) & (raw.international_transaction == 0)).sum(),
    # card behaviour windows (a 1-hour window sits inside the 24-hour window)
    "transactions_last_1h > transactions_last_24h":       (raw.transactions_last_1h > raw.transactions_last_24h).sum(),
    "transactions_last_24h > transactions_last_7d":       (raw.transactions_last_24h > raw.transactions_last_7d).sum(),
    "amount_last_1h > amount_last_24h":                   (raw.amount_last_1h > raw.amount_last_24h + 0.01).sum(),
    # card presence
    "chip used but card not present":                     ((raw.used_chip == 1) & (raw.card_present == 0)).sum(),
    "PIN used but card not present":                      ((raw.used_pin == 1) & (raw.card_present == 0)).sum(),
    # fraud label vs. card-presence flag
    "fraud_type 'Card-Not-Present' but card_present = 1": ((raw.fraud_type == "Card-Not-Present") & (raw.card_present == 1)).sum(),
    "fraud_type 'Card-Present' but card_present = 0":     ((raw.fraud_type == "Card-Present") & (raw.card_present == 0)).sum(),
    # customer attributes should be fixed per customer
    "customers with more than one age/gender/state":      int((raw.groupby("customer_id")[["customer_age", "customer_gender", "customer_state"]].nunique() > 1).any(axis=1).sum()),
}
cons = pd.Series(rules, name="rows breaking rule").to_frame()
cons["status"] = np.where(cons["rows breaking rule"] == 0, "OK", "CHECK")
cons

In [ ]:
# Look closer at the rules that fail
print("Reported weekday (columns) vs actual weekday of the date (rows):")
display(pd.crosstab(dt.dt.day_name(), raw.transaction_day_of_week))

print("\nCard-presence flags by channel (share of rows):")
display(raw.groupby("transaction_channel")[["card_present", "used_chip", "used_pin"]].mean().round(2))

bad = raw.amount_last_1h > raw.amount_last_24h + 0.01
print(f"\namount_last_1h > amount_last_24h: {bad.sum():,} rows ({bad.mean():.1%}); "
      f"fraud rate in those rows {raw.is_fraud[bad].mean()*100:.2f}% vs {raw.is_fraud.mean()*100:.3f}% overall")

**What the consistency checks found.** Most rules hold perfectly: the outcome fields agree with each other, the hour matches the timestamp, international status matches the merchant country, and each customer's attributes never change. Four problems do turn up, and they weren't mentioned in the original report:

1. **`transaction_day_of_week` does not match the actual date.** Not a single row agrees with the weekday of `transaction_datetime`; the labels look shifted by a few days. `is_weekend` agrees with the (wrong) day-of-week column, so it's also out of line with the real calendar. *Impact:* both features are effectively noise. Fraud rates are flat across days (about 1.9–2.2%), neither feature is among the top 10 drivers, and recomputing both from the timestamp or dropping them leaves test PR-AUC essentially unchanged. Because the columns are both wrong and uninformative, **we drop both from the model** (Section 3) rather than guess whether the label or the timestamp is right.
2. **ATM transactions are marked `card_present = 0` even though the PIN was used.** A card has to be in an ATM to enter a PIN, so the flag is mislabeled for those 5,932 rows. *Impact:* small. The model also receives `transaction_channel` and `device_type`, which identify ATM rows exactly.
3. **`amount_last_1h` exceeds `amount_last_24h` in 4,593 rows (4.6%).** A 1-hour window can't contain more spending than the 24-hour window around it. These look like simulation noise. *Impact:* small. We keep the rows rather than drop 4.6% of the data, because both features remain informative and dropping them would bias the sample.
4. **`fraud_type` sometimes disagrees with `card_present`.** 36 frauds labeled "Card-Not-Present" have the card present, and 103 labeled "Card-Present" do not. *Impact:* none on the model, because `fraud_type` is an outcome label that is never a model input. It only affects how losses are split by type (Charter Figure 1, Report Figure B4).

All four are consistent with the dataset being simulated. **We keep every record rather than filtering:** Section 9 ends with a robustness check showing that fixing these fields, or dropping the affected rows, leaves the conclusions unchanged.

### Dates later than the charter

The file covers all of 2026, but the project charter was written on October 24, 2026. Any transaction dated after that cannot be a real record, so counting them tests whether the data are simulated.

In [ ]:
CHARTER_DATE = pd.Timestamp("2026-10-24")
late = raw.transaction_datetime >= CHARTER_DATE + pd.Timedelta(days=1)
R["after_charter_date"] = {"rows": int(late.sum()), "share": float(late.mean())}
print(f"Transactions dated after {CHARTER_DATE:%B %d, %Y}: {late.sum():,} ({late.mean():.1%})")
print("Latest transaction:", raw.transaction_datetime.max())

**Result:** 18,601 transactions (18.6%) are dated after October 24, 2026, running to December 31. Together with the consistency problems above, this confirms the data are simulated. The time-based split still works inside the simulated timeline, but results show the method, not what a real bank would see.

### Target variable: how rare is fraud, and is the rate stable?

**Why:** class imbalance drives three later choices (balanced class weights, PR-AUC as the metric, a cost-based threshold). And because we split by month, we need to know whether the fraud rate drifts across the year; if January had 5% fraud and December 1%, a time split would be testing on a different problem.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1, 2]})
counts = raw.is_fraud.value_counts().sort_index()
ax[0].bar(["Legitimate", "Fraud"], counts.values, color=[GREY, RED])
for i, v in enumerate(counts.values):
    ax[0].text(i, v, f"{v:,}\n({v/len(raw):.2%})", ha="center", va="bottom", fontsize=9)
ax[0].set_title("Class balance"); ax[0].set_ylim(0, counts.max() * 1.25)

m = raw.groupby(dt.dt.month).agg(n=("is_fraud", "size"), rate=("is_fraud", "mean"))
ax[1].plot(m.index, m.rate * 100, color=NAVY, marker="o")
ax[1].axhline(raw.is_fraud.mean() * 100, color=GREY, ls="--", lw=1)
ax[1].axvspan(8.5, 9.5, color=GOLD, alpha=.15); ax[1].axvspan(9.5, 12.5, color=RED, alpha=.08)
ax[1].text(9, ax[1].get_ylim()[1] * .98, "valid", ha="center", va="top", fontsize=8)
ax[1].text(11, ax[1].get_ylim()[1] * .98, "test", ha="center", va="top", fontsize=8)
ax[1].set_xticks(range(1, 13)); ax[1].set_xlabel("Month of 2026"); ax[1].set_ylabel("Fraud rate (%)")
ax[1].set_title("Fraud rate by month")
save("qc_target.png")
m.assign(rate=lambda t: (t.rate * 100).round(2)).T

**Result:** about 49 legitimate transactions for every fraud, a severe imbalance. Monthly volume is steady (about 7,900–8,600) and the fraud rate stays in a narrow band around 2%. The year is stable enough that a train-on-the-past, test-on-the-future split is a fair test.

### Distributions of the numeric features

**Why:** the shape of each feature affects the model. Logistic regression is sensitive to heavily skewed inputs, because a few extreme values pull the fitted line. Gradient boosting is not, because it splits on rank order. We plot each continuous feature, with fraud and legitimate transactions overlaid, and measure **skewness** (0 = symmetric; above about 1 = strongly right-skewed).

In [ ]:
cont = [c for c in raw.select_dtypes("number").columns
        if raw[c].nunique() > 10 and c not in ["fraud_amount_lost"]]
ncol = 4; nrow = int(np.ceil(len(cont) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(13, 2.4 * nrow))
for a, c in zip(axes.flat, cont):
    a.hist(raw.loc[raw.is_fraud == 0, c], bins=40, density=True, color=GREY, alpha=.6, label="Legit")
    a.hist(raw.loc[raw.is_fraud == 1, c], bins=40, density=True, color=RED,  alpha=.6, label="Fraud")
    a.set_title(f"{c}\nskew = {raw[c].skew():.1f}", fontsize=9); a.tick_params(labelsize=7)
for a in axes.flat[len(cont):]:
    a.axis("off")
axes.flat[0].legend(fontsize=7, frameon=False)
save("qc_distributions.png")

raw[cont].skew().sort_values(ascending=False).round(1).to_frame("skewness")

**Result:** the money-related features are strongly right-skewed: `amount_vs_avg_ratio` (skew about 29), `transaction_amount` (about 15), `amount_last_1h` (about 9). Most values are small and a few are huge. Hour, credit score, age and account features are close to symmetric or only mildly skewed.

**What we decided:** we did **not** log-transform the skewed features, for three reasons. Gradient boosting doesn't need it. Standardization plus regularization (`C=0.5`) already limits the pull of extreme values on logistic regression. And logistic regression still won the model comparison. A log transform is a reasonable improvement to try in future work.

### Outliers

We use the standard **IQR rule**: a value is an outlier if it is more than 1.5 × the interquartile range below the 25th percentile or above the 75th. The more important question is what to *do* with them.

In [ ]:
num = raw[cont]
q1, q3 = num.quantile(.25), num.quantile(.75); iqr = q3 - q1
is_out = (num < q1 - 1.5 * iqr) | (num > q3 + 1.5 * iqr)
out = pd.DataFrame({"outlier rows": is_out.sum(),
                    "pct": (is_out.mean() * 100).round(1),
                    "fraud rate among outliers %": [round(raw.is_fraud[is_out[c]].mean() * 100, 2) if is_out[c].any() else np.nan for c in cont]})
print(f"Overall fraud rate: {raw.is_fraud.mean()*100:.3f}%")
out.sort_values("outlier rows", ascending=False)

**Result and decision:** several features have many outliers by the IQR rule, which is expected for skewed distributions. We **keep all of them**. In fraud detection, unusual transactions are often exactly the ones we're looking for; the last column shows that outlying values frequently carry a fraud rate above the 2.06% average. Removing outliers would delete signal, not noise. None of the values is impossible (checked earlier), so there is no data-entry error to correct.

### Correlation between features

**Why:** when two features carry nearly the same information (multicollinearity), logistic regression struggles to split credit between them. Predictions stay fine, but individual coefficients become unstable and hard to interpret. We list the most strongly correlated pairs.

In [ ]:
nm = raw.select_dtypes("number").drop(columns=["is_fraud", "fraud_amount_lost"])
corr = nm.corr()
pairs = (corr.abs().where(np.triu(np.ones(corr.shape), 1).astype(bool))
             .stack().sort_values(ascending=False).head(10).round(2))

plt.figure(figsize=(9, 7.5))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(fraction=.03, label="Pearson correlation")
plt.xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(corr)), corr.columns, fontsize=7)
plt.title("Correlation between numeric features")
save("qc_correlation.png")
pairs.to_frame("|r|")

**Result:** the strongest pairs make sense: a card's 30-day average and 30-day standard deviation (r = .94), card present and chip used (.91), distance from home and international (.78), credit limit and credit score (.72). No pair is a perfect duplicate.

**Decision:** we keep all of them. The L2 penalty in logistic regression (`C=0.5`) is designed to handle correlated inputs by sharing weight between them; gradient boosting is unaffected; and the report evaluates the model on predictions and permutation importance rather than on individual coefficients.

### Data quality summary

| Check | Result | Action |
|---|---|---|
| Missing values | Only `browser`; all 41,709 blanks are ATM/In-store | Fill with "None" (Section 1c) |
| Duplicates | None | None needed |
| Impossible values | None; a few values sit at generator caps | None needed |
| Category spelling | Clean | None needed |
| Outcome fields | Fully consistent | None needed |
| Day of week / weekend | Don't match the actual date; no fraud signal | Dropped from the model |
| ATM `card_present` | Mislabeled 0 despite PIN use | Keep; channel and device identify ATM |
| `amount_last_1h` > `amount_last_24h` | 4.6% of rows | Keep; note as simulation noise |
| `fraud_type` vs `card_present` | 139 frauds disagree | Keep; outcome label only, not a model input |
| Dates after the charter | 18.6% of rows dated after October 24, 2026 | Confirms the data are simulated; report as a limitation |
| Class balance | 2.06% fraud, stable by month | Balanced weights, PR-AUC, time split |
| Skew | Money features strongly skewed | Standardize + regularize; log transform as future work |
| Outliers | Many, often fraud-rich | Keep: they carry signal |
| Correlation | Several strong but sensible pairs | Keep; L2 penalty handles it |

The data are clean enough to model. Now we apply the cleaning steps.

## 1c. Clean the data

**What the next cell does and why:**
- **Parses `transaction_datetime` as a date** so we can split by month later.
- **Fills the blank `browser` field with `"None"`.** All 41,709 blanks are in-store or ATM transactions, where no browser exists. The value is *structurally* absent, not missing at random, so imputing a browser would be wrong and dropping the rows would throw away 42% of the data. "None" is the true answer.
- **Engineers `night_0_4`** (1 if the hour is midnight to 4 a.m.). Fraud spikes overnight, and a logistic regression treats `transaction_hour` as a straight line (hour 23 is "bigger" than hour 2), so it can't see a spike at one end of the clock without this flag. **The window was chosen from the training months only** (January–August), so the test months played no part in designing it; Section 2 shows that check.
- **Checks for duplicates and computes the fraud base rate.** The base rate (2.06%) is the benchmark every later metric is compared with.

In [ ]:
df = raw.copy()                                               # clean a copy; raw stays untouched
df["browser"] = df["browser"].fillna("None")                 # no browser for POS/ATM
df["month"] = df["transaction_datetime"].dt.month
df["night_0_4"] = (df["transaction_hour"] <= 4).astype(int)  # engineered after EDA

R.update({"rows": len(df), "cols": df.shape[1],
          "customers": int(df.customer_id.nunique()), "cards": int(df.card_id.nunique()),
          "merchants": int(df.merchant_id.nunique()),
          "start": str(df.transaction_datetime.min().date()),
          "end": str(df.transaction_datetime.max().date()),
          "missing_browser": int((df.browser == "None").sum()),
          "duplicates": int(df.transaction_id.duplicated().sum()),
          "fraud_n": int(df.is_fraud.sum()), "fraud_rate": float(df.is_fraud.mean()),
          "loss_total": float(df.fraud_amount_lost.sum())})

print(f"Rows x columns : {df.shape}  (43 original columns + month + night_0_4)")
print(f"Date range     : {R['start']} to {R['end']}")
print(f"Customers / cards / merchants: {R['customers']:,} / {R['cards']:,} / {R['merchants']:,}")
print(f"Duplicate IDs  : {R['duplicates']}")
print(f"Browser = None : {R['missing_browser']:,}")
print(f"Fraud          : {R['fraud_n']:,} transactions ({R['fraud_rate']*100:.3f}%), ${R['loss_total']:,.0f} lost")
print("\nRemaining missing values per column (should be all zero):")
print(df.isna().sum()[df.isna().sum() > 0])

## 2. Exploratory analysis

**Why look before modeling:** EDA tells us which signals exist, whether any need engineering, and gives the decision owner intuition the model alone can't. It is also where we spotted the overnight spike that became `night_0_4`.

**Why fraud *rate* bar charts:** fraud is 2% of rows, so raw counts mostly show which groups are *big*. The rate shows which groups are *risky*. The dashed line marks the overall rate, and the riskiest bar is red.

In [ ]:
def rate_bar(col, fname, title, order=None):
    g = df.groupby(col).is_fraud.mean().mul(100)
    g = g.reindex(order) if order else g.sort_values()
    plt.figure(figsize=(6.5, 0.35 * len(g) + 1.2))
    colors = [RED if v == g.max() else NAVY for v in g.values]
    plt.barh([str(i) for i in g.index], g.values, color=colors)
    for i, v in enumerate(g.values):
        plt.text(v + g.max() * 0.01, i, f"{v:.1f}%", va="center", fontsize=9)
    plt.axvline(df.is_fraud.mean() * 100, color=GREY, ls="--", lw=1)
    plt.xlabel("Fraud rate (%)  |  dashed line = overall rate")
    save(fname)
    return {str(k): round(float(v), 2) for k, v in g.items()}

### Report Figure 1: Fraud rate by country risk

We combine two flags into one three-level "risk profile" so the gradient is easy to read: domestic → international → high-risk country. **Result:** 0.35% → 2.7% → 11.1%. This is also the basis for the simple **benchmark rule** ("flag every high-risk-country transaction") that the model has to beat.

In [ ]:
df["Risk profile"] = np.select(
    [df.high_risk_country == 1, df.international_transaction == 1],
    ["High-risk country", "International (other)"], "Domestic")
R["rate_risk"] = rate_bar("Risk profile", "fig_risk.png", "Fraud rate by country risk",
                          order=["Domestic", "International (other)", "High-risk country"])
R["rate_risk"]

### Report Figure 2: Fraud rate by hour of day

A line chart because hour is ordered. **Result:** about 2% in the daytime, rising to roughly 11% at 3 a.m. This is the chart that led us to add `night_0_4`, which later became the fourth most important model driver.

In [ ]:
h = df.groupby("transaction_hour").is_fraud.mean().mul(100)
plt.figure(figsize=(6.5, 3.2))
plt.plot(h.index, h.values, color=NAVY, marker="o", ms=3)
plt.axhline(df.is_fraud.mean() * 100, color=GREY, ls="--", lw=1)
plt.xlabel("Hour of day"); plt.ylabel("Fraud rate (%)"); plt.xticks(range(0, 24, 3))
save("fig_hour.png")
R["rate_hour_max"] = [int(h.idxmax()), round(float(h.max()), 2)]
R["rate_hour_min"] = [int(h.idxmin()), round(float(h.min()), 2)]
print("Highest hour:", R["rate_hour_max"], "| Lowest hour:", R["rate_hour_min"])

### Designing the night flag from training data only

Figure 2 uses the whole year because it describes the data. But any feature we *design* from a chart must not use the test months (October–December); otherwise the test set has influenced the model and is no longer an independent check. So the night window is chosen from **January–August only**: we flag the consecutive overnight hours whose fraud rate in the training months is clearly above the training-month average.

In [ ]:
tr_h = df[df.month <= 8].groupby("transaction_hour").is_fraud.mean().mul(100)
avg = df[df.month <= 8].is_fraud.mean() * 100
night = df[df.month <= 8].query("transaction_hour <= 4").is_fraud.mean() * 100
other = df[df.month <= 8].query("transaction_hour > 4").is_fraud.mean() * 100
R["night_check_train"] = {"rate_0_4": float(night), "rate_other": float(other),
                          "peak_hour": int(tr_h.idxmax()), "peak_rate": float(tr_h.max())}
print(f"Training months only: fraud rate {night:.2f}% for hours 0-4 vs {other:.2f}% for other hours "
      f"(average {avg:.2f}%); peak at {tr_h.idxmax()}:00 with {tr_h.max():.1f}%")
print("Hours 0-6 in training months:", tr_h.loc[0:6].round(2).to_dict())

### Report Figures B1 and B2: Fraud rate by channel and by merchant category

**Result:** online (3.2%) is about four times riskier than in-store (0.8%); online retail and travel lead the categories.

In [ ]:
R["rate_channel"]  = rate_bar("transaction_channel", "fig_channel.png",  "Fraud rate by transaction channel")
R["rate_category"] = rate_bar("merchant_category",   "fig_category.png", "Fraud rate by merchant category")

### Report Figure B3: Transaction amount, fraud vs. legitimate

**Why a log scale and density:** amounts are heavily right-skewed (a few very large purchases), so a linear axis squashes most transactions into one bar. Density (rather than counts) lets the small fraud group and the large legitimate group be compared on the same scale.

**Result:** the two distributions overlap heavily (medians about \$65 vs. \$52), so amount *alone* is a weak signal. That's one reason we need a multi-feature model rather than a single rule.

In [ ]:
plt.figure(figsize=(6.5, 3.2))
bins = np.logspace(0, np.log10(df.transaction_amount.max() + 1), 40)
plt.hist(df[df.is_fraud == 0].transaction_amount, bins=bins, density=True, alpha=0.6, color=GREY, label="Legitimate")
plt.hist(df[df.is_fraud == 1].transaction_amount, bins=bins, density=True, alpha=0.7, color=RED,  label="Fraud")
plt.xscale("log"); plt.xlabel("Transaction amount (USD, log scale)"); plt.ylabel("Density")
plt.legend(frameon=False)
save("fig_amount.png")
R["median_amt_fraud"] = float(df[df.is_fraud == 1].transaction_amount.median())
R["median_amt_legit"] = float(df[df.is_fraud == 0].transaction_amount.median())
print(f"Median amount: fraud ${R['median_amt_fraud']:.2f} vs legitimate ${R['median_amt_legit']:.2f}")

### Report Figure B4 / Charter Figure 1: Fraud losses by fraud type

Sum of `fraud_amount_lost` within each `fraud_type`, fraud rows only. **Result:** card-not-present fraud is \$150,504 of the \$279,842 total (53.8%), more than the other three types combined, which is why the policy focuses on transaction-time signals that work for online purchases.

*Note:* `fraud_type` and `fraud_amount_lost` are **outcomes**. We use them here to describe the problem and later to put dollar weights on the policy, but **never** as model inputs (see Section 3).

In [ ]:
ft = (df[df.is_fraud == 1].groupby("fraud_type")
        .agg(n=("is_fraud", "size"), loss=("fraud_amount_lost", "sum"))
        .sort_values("loss"))
plt.figure(figsize=(6.5, 2.8))
plt.barh(ft.index, ft.loss / 1000, color=NAVY)
for i, (n, l) in enumerate(zip(ft.n, ft.loss)):
    plt.text(l / 1000 + 1, i, f"${l/1000:,.0f}K  ({n} cases)", va="center", fontsize=9)
plt.xlabel("Total loss (USD thousands)")
plt.xlim(0, ft.loss.max() / 1000 * 1.35)
save("fig_fraudtype.png")
R["loss_by_type"] = {k: round(float(v), 0) for k, v in ft.loss.items()}
ft.assign(share_pct=(ft.loss / ft.loss.sum() * 100).round(1))

## 3. Choose the features (and guard against leakage)

**What:** decide which columns the model is allowed to see.

**Why each exclusion:**
- **Identifiers** (`transaction_id`, `customer_id`, `card_id`, `merchant_id`) are arbitrary codes. A model could memorize them, which wouldn't generalize to new customers.
- **`transaction_datetime` and `month`**: the split is by time, so the date would just tell the model which set a row is in. The useful time signals (hour and the night flag) are kept separately.
- **`transaction_day_of_week` and `is_weekend`**: Section 1b showed these contradict the transaction dates (day of week is wrong on every row, weekend on 57,158 rows), and the fraud rate is nearly flat across days. They are wrong *and* carry no signal, so we drop them rather than guess which column is right.
- **Outcome fields** (`is_fraud`, `fraud_type`, `fraud_amount_lost`) are only known *after* fraud is confirmed. Using them would be **leakage**: perfect scores in testing, useless in real life.
- **`customer_gender` and `customer_age`** are protected attributes. We exclude them on ethical grounds and use them **only** in the fairness audit (Section 10).
- **`Risk profile`** was built for the EDA chart; its two underlying flags are already features.

Everything left is something a fraud system would actually know **at the moment of purchase**. That gives **32 features**: 25 numeric and 7 categorical.

In [ ]:
DROP = ["transaction_id", "customer_id", "card_id", "merchant_id",
        "transaction_datetime", "month",
        "is_fraud", "fraud_type", "fraud_amount_lost",       # outcomes -> leakage
        "customer_gender", "customer_age",                    # protected -> fairness audit only
        "transaction_day_of_week", "is_weekend",              # contradict the dates, no signal
        "Risk profile"]                                       # EDA helper
CAT = ["merchant_category", "merchant_country",
       "customer_state", "transaction_channel", "device_type", "browser", "email_domain"]
NUM = [c for c in df.columns if c not in DROP + CAT]

R["n_features"] = len(NUM) + len(CAT)
R["features"] = NUM + CAT
print(f"{len(NUM)} numeric + {len(CAT)} categorical = {R['n_features']} features\n")
print("Numeric:", NUM, "\n")
print("Categorical:", CAT)

## 4. Time-based split: train Jan–Aug, validate Sep, test Oct–Dec

**Why split by time instead of randomly:** in practice the model is trained on the past and used on the *future*. A random split mixes December transactions into training and lets the model "see the future", which usually makes results look better than they will be in deployment. A time split is the honest test, and it also reveals drift if fraud patterns change.

**Why three sets:**
- **Train (Jan–Aug):** fit the models.
- **Validation (Sep):** make every choice: which model, and which threshold.
- **Test (Oct–Dec):** touched **once**, after every choice is final. Because nothing was chosen on it, its results are an unbiased estimate of performance on later months.

**Nothing is tuned.** Both models' settings were fixed in advance. September is used only to choose between the two models and to choose the review threshold. The final model is the one trained on January–August; it is not refit after September.

In [ ]:
train = df[df.month <= 8]
valid = df[df.month == 9]
test  = df[df.month >= 10]

R["split"] = {k: {"n": len(v), "fraud": int(v.is_fraud.sum())}
              for k, v in [("train", train), ("valid", valid), ("test", test)]}
Xtr, ytr = train[NUM + CAT], train.is_fraud
Xva, yva = valid[NUM + CAT], valid.is_fraud
Xte, yte = test[NUM + CAT],  test.is_fraud

pd.DataFrame(R["split"]).T.assign(fraud_rate=lambda t: (t.fraud / t.n).map("{:.2%}".format))

## 5. Train two models and pick one on validation PR-AUC

We compare a simple, explainable model with a flexible one, and let the data decide.

**Model A: Logistic regression**
- *Why:* fast, stable, and its coefficients can be explained to reviewers and regulators.
- `StandardScaler` puts numeric features on the same scale so the penalty treats them equally.
- `OneHotEncoder(min_frequency=20)` turns categories into 0/1 columns and groups very rare categories together so they don't add noise.
- `class_weight="balanced"` makes each fraud case count about 48× more than a legitimate one. Without it, the model can get 98% accuracy by predicting "never fraud".
- `C=0.5` adds mild regularization to reduce overfitting.

**Model B: Histogram gradient boosting**
- *Why:* it captures non-linear patterns and interactions automatically, and is usually a strong performer on tabular data, so it's the natural challenger.
- It splits on categories natively (`categorical_features`), so ordinal encoding is enough.
- Small learning rate (0.05), minimum leaf size of 40 and L2 penalty all limit overfitting.
- **Early stopping is on.** scikit-learn switches it on automatically for more than 10,000 rows: it holds out a random 10% of the training data and stops adding trees once that slice stops improving for 10 rounds. So `max_iter=300` is a ceiling; the cell below prints how many trees were actually built.
- It uses no class weights (the default), unlike logistic regression. PR-AUC depends only on how transactions are ranked, so this matters less than it would for accuracy, but it is a difference worth stating.

**Why judge by PR-AUC rather than accuracy or ROC-AUC:** with 2% fraud, accuracy is meaningless, and ROC curves can look optimistic because the huge pool of legitimate transactions makes the false-positive *rate* look small (Saito & Rehmsmeier, 2015). PR-AUC asks the business question directly: *among the transactions we flag, how many are actually fraud, at each level of recall?* A no-skill model's PR-AUC equals the base rate (about 0.02).

> Training takes roughly 30–60 seconds.

In [ ]:
logit = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT)])),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5))])

gbm = Pipeline([
    ("prep", ColumnTransformer([
        ("num", "passthrough", NUM),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT)])),
    ("clf", HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40,
        l2_regularization=1.0,
        categorical_features=list(range(len(NUM), len(NUM) + len(CAT))),
        random_state=SEED))])

models = {"Logistic regression": logit, "Gradient boosting": gbm}
for name, m in models.items():
    m.fit(Xtr, ytr)
    print("trained:", name)

R["training"] = {"logit_iterations": int(logit[-1].n_iter_[0]),
                 "logit_input_columns": int(logit[-1].coef_.shape[1]),
                 "gbm_trees": int(gbm[-1].n_iter_),
                 "gbm_early_stopping": bool(gbm[-1].do_early_stopping_)}
print(f"Logistic regression: {R['training']['logit_input_columns']} input columns after encoding, "
      f"converged in {R['training']['logit_iterations']} iterations")
print(f"Gradient boosting: early stopping = {R['training']['gbm_early_stopping']}, "
      f"stopped at {R['training']['gbm_trees']} of 300 trees")

R["valid"] = {}
for name, m in models.items():
    p_va = m.predict_proba(Xva)[:, 1]
    R["valid"][name] = {"pr": average_precision_score(yva, p_va), "roc": roc_auc_score(yva, p_va)}

best_name = max(R["valid"], key=lambda n: R["valid"][n]["pr"])
best = models[best_name]
R["best_model"] = best_name

print(f"\nNo-skill PR-AUC on validation = base rate = {yva.mean():.3f}")
print("Selected on validation PR-AUC:", best_name)
pd.DataFrame(R["valid"]).T.round(3).rename(columns={"pr": "Validation PR-AUC", "roc": "Validation ROC-AUC"})

**Reading the result (Report Table 1):** gradient boosting has a slightly higher ROC-AUC, but logistic regression wins on PR-AUC (.180 vs .137), the metric we committed to in advance, so we select it. It is also the easier model to explain. The lesson: the more complex model is not automatically better on the metric that matters.

## 6. Choose the review threshold by net savings (on September)

**What:** the model outputs a score from 0 to 1. Every transaction at or above a threshold is held for review. We try thresholds from 0.02 to 0.95 and pick the one with the highest **net savings**:

$$\text{net savings} = \underbrace{0.80 \times \sum \text{losses on frauds we flagged}}_{\text{expected losses prevented}} \; - \; \underbrace{\$5 \times \text{number flagged}}_{\text{review cost}}$$

The 0.80 is the **review success rate**: we assume a review prevents 80% of the loss on a flagged fraud, because reviewers miss some fraud. Because both the 0.80 and the \$5 are assumptions, the result is an *estimate* of net savings.

**Why cost-based instead of the default 0.5:** 0.5 is arbitrary. The right threshold depends on what a review costs and what a missed fraud costs, which is the core idea of cost-sensitive learning (Elkan, 2001). Using real fraud dollars as the benefit also means the model gets credit for catching *expensive* frauds.

**Why on validation, not test:** if we picked the threshold on the test set, the test result would be optimistically biased.

In [ ]:
def net_savings(flag, y, loss, cost=REVIEW_COST, success=REVIEW_SUCCESS):
    """Estimated net savings of any review policy (a True/False flag per transaction):
    expected fraud losses prevented minus review cost."""
    return float(success * loss[flag & (y == 1)].sum() - cost * flag.sum())

def savings(y, p, loss, t, cost=REVIEW_COST, success=REVIEW_SUCCESS):
    """Net savings of the model policy that flags every score >= t."""
    return net_savings(p >= t, y, loss, cost, success)

pva  = best.predict_proba(Xva)[:, 1]
grid = np.round(np.arange(0.02, 0.96, 0.01), 2)
sv   = [savings(yva.values, pva, valid.fraud_amount_lost.values, t) for t in grid]
thr  = float(grid[int(np.argmax(sv))])
R["threshold"] = thr

# Report Figure 3
plt.figure(figsize=(6.5, 3.2))
plt.plot(grid, np.array(sv) / 1000, color=NAVY, lw=2)
plt.axvline(thr, color=RED, ls="--", lw=1)
plt.text(thr + 0.01, max(sv) / 1000 * 0.95, f"Best threshold = {thr:.2f}", color=RED, fontsize=9)
plt.xlabel("Model score threshold"); plt.ylabel("Estimated net savings (USD thousands)")
save("fig_threshold.png")
print(f"Chosen threshold: {thr:.2f}  (validation net savings ${max(sv):,.0f})")

**Remember:** 0.79-type cutoffs are *ranking scores*, not probabilities. Section 9 shows that a score near 0.8 means roughly a 10% chance of fraud, so the cutoff should never be read as "79% likely to be fraud."

**Reading Figure 3:** at low thresholds we flag so many legitimate transactions that review costs exceed recovered losses (savings go negative). At very high thresholds we miss most fraud. The peak, 0.79, balances the two.

## 7. Evaluate once on the held-out test set (Oct–Dec)

Everything is now locked: model, features and threshold. We score the test set **once** and compare the policy with the simple benchmark rule.

**Why compare with a rule:** a model is only worth deploying if it beats what a fraud team could do without one. "Flag every high-risk-country transaction" is the obvious simple alternative from the EDA (Figure 1). This is hypothesis **H2**.

### The baseline rule, and why it involves no leakage

The rule flags every transaction whose merchant country carries the dataset's `high_risk_country` flag. The charter promises three things about it, and the next cell checks each one:
1. **It depends only on the merchant country**, so it is known at the moment of purchase.
2. **The country list is the same in every month**, so applying it to October–December uses nothing learned from those months.
3. **The list comes from the dataset**, not from the team picking the countries with the most fraud. The rule has no settings, so there is nothing to tune.

In [ ]:
hr = df.groupby("merchant_country")["high_risk_country"].agg(["min", "max"])
HIGH_RISK = sorted(hr.index[hr["max"] == 1])
fixed_by_country = bool((hr["min"] == hr["max"]).all())
lists_by_month = (df[df.high_risk_country == 1].groupby("month")["merchant_country"]
                    .apply(lambda c: tuple(sorted(c.unique()))))
same_every_month = lists_by_month.nunique() == 1 and len(lists_by_month) == 12

R["baseline_rule"] = {"countries": HIGH_RISK, "fixed_by_country": fixed_by_country,
                      "same_every_month": bool(same_every_month)}
print("High-risk countries in the dataset :", ", ".join(HIGH_RISK))
print("Flag depends only on country       :", fixed_by_country)
print("Same list in all 12 months         :", same_every_month)

In [ ]:
pte = {n: m.predict_proba(Xte)[:, 1] for n, m in models.items()}
p = pte[best_name]
y = yte.values
loss_te = test.fraud_amount_lost.values
flag = p >= thr
tn, fp, fn, tp = confusion_matrix(y, flag).ravel()

R["base_rate_test"] = float(y.mean())
R["test"] = {n: {"pr": average_precision_score(y, pp), "roc": roc_auc_score(y, pp)} for n, pp in pte.items()}
R["test_policy"] = {
    "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    "flagged": int(flag.sum()), "flag_rate": float(flag.mean()),
    "precision": float(tp / max(tp + fp, 1)), "recall": float(tp / (tp + fn)),
    "loss_total": float(loss_te.sum()), "loss_caught": float(loss_te[flag & (y == 1)].sum()),
    "loss_prevented": float(REVIEW_SUCCESS * loss_te[flag & (y == 1)].sum()),
    "review_cost": float(REVIEW_COST * flag.sum()),
    "net_savings": savings(y, p, loss_te, thr)}
R["test_policy"]["loss_caught_pct"] = R["test_policy"]["loss_caught"] / R["test_policy"]["loss_total"]

# Benchmark rule: flag every high-risk-country transaction
rule = test.high_risk_country.values == 1
R["rule"] = {"flagged": int(rule.sum()), "caught": int((rule & (y == 1)).sum()),
             "recall": float((rule & (y == 1)).sum() / y.sum()),
             "precision": float((rule & (y == 1)).sum() / rule.sum()),
             "loss_caught": float(loss_te[rule & (y == 1)].sum()),
             "loss_prevented": float(REVIEW_SUCCESS * loss_te[rule & (y == 1)].sum()),
             "review_cost": float(REVIEW_COST * rule.sum()),
             "net_savings": net_savings(rule, y, loss_te)}

# ---- Report Table 1 (full) ----
t1 = pd.DataFrame({n: {"Validation PR-AUC": R["valid"][n]["pr"], "Validation ROC-AUC": R["valid"][n]["roc"],
                       "Test PR-AUC": R["test"][n]["pr"], "Test ROC-AUC": R["test"][n]["roc"]} for n in models}).T
t1.loc["No-skill baseline"] = [yva.mean(), 0.5, y.mean(), 0.5]
print("TABLE 1: Model performance"); display(t1.round(3))

lift = R["test"][best_name]["pr"] / y.mean()
print(f"H1 check: test PR-AUC is {lift:.1f}x the base rate (success bar was 3x) -> {'SUPPORTED' if lift >= 3 else 'not supported'}")

In [ ]:
# ---- Report Table 2: policy vs rule ----
tp_, r_ = R["test_policy"], R["rule"]
nf = int(y.sum())
t2 = pd.DataFrame({
    f"Model policy (threshold {thr:.2f})": [
        f"{tp_['flagged']:,} ({tp_['flag_rate']:.1%})",
        f"{tp_['tp']} of {nf} ({tp_['recall']:.1%})",
        f"{tp_['precision']:.1%}",
        f"${tp_['loss_caught']:,.0f} of ${tp_['loss_total']:,.0f} ({tp_['loss_caught_pct']:.1%})",
        f"${tp_['loss_prevented']:,.0f}",
        f"${tp_['review_cost']:,.0f}", f"${tp_['net_savings']:,.0f}"],
    "Rule: flag high-risk countries": [
        f"{r_['flagged']:,} ({r_['flagged']/len(y):.1%})",
        f"{r_['caught']} of {nf} ({r_['recall']:.1%})",
        f"{r_['precision']:.1%}",
        f"${r_['loss_caught']:,.0f} of ${tp_['loss_total']:,.0f} ({r_['loss_caught']/tp_['loss_total']:.1%})",
        f"${r_['loss_prevented']:,.0f}",
        f"${r_['review_cost']:,.0f}", f"${r_['net_savings']:,.0f}"]},
    index=["Transactions flagged", "Fraud cases caught (recall)", "Precision of flags",
           "Fraud dollars on flagged frauds", f"Expected losses prevented ({REVIEW_SUCCESS:.0%} review success)",
           f"Review cost at ${REVIEW_COST:.0f}", "Estimated net savings"])
print("TABLE 2: Review policy outcomes, Oct-Dec 2026"); display(t2)
print(f"About {tp_['flagged']/92:.0f} reviews per day over the 92-day quarter.")

### Report Figure 4: Precision-recall curves on the test set

Each curve shows the trade-off between catching more fraud (recall, x-axis) and keeping flags accurate (precision, y-axis). The dashed line is a model with no skill. The red dot is where our chosen threshold lands.

In [ ]:
plt.figure(figsize=(6.5, 3.6))
for (nme, pp), col in zip(pte.items(), [GREY, NAVY]):
    pr, rc, _ = precision_recall_curve(y, pp)
    plt.plot(rc, pr, color=col, lw=2, label=f"{nme} (PR-AUC {average_precision_score(y, pp):.2f})")
plt.axhline(y.mean(), color=GREY, ls="--", lw=1, label=f"No-skill baseline ({y.mean():.3f})")
plt.scatter([tp_["recall"]], [tp_["precision"]], color=RED, zorder=5, label=f"Chosen threshold ({thr:.2f})")
plt.xlabel("Recall (share of fraud caught)"); plt.ylabel("Precision (share of flags that are fraud)")
plt.legend(frameon=False, fontsize=8)
save("fig_pr_curve.png")

### Report Figure B6: Confusion matrix at the chosen threshold

**Why show it:** it turns rates into counts the fraud team can picture. The **false positives** (legitimate but flagged) are the main cost of the policy, but because flagged transactions go to a human reviewer rather than being declined, each one costs analyst time, not a lost sale. The **false negatives** (missed fraud) show the model is a triage tool, not a complete defense.

In [ ]:
cm = np.array([[tn, fp], [fn, tp]])
plt.figure(figsize=(4.2, 3.4))
plt.imshow(cm, cmap="Blues")
for (a, b), v in np.ndenumerate(cm):
    plt.text(b, a, f"{v:,}", ha="center", va="center", fontsize=13, fontweight="bold",
             color="white" if v > cm.max() / 2 else "black")
plt.xticks([0, 1], ["Not flagged", "Flagged"]); plt.yticks([0, 1], ["Legitimate", "Fraud"])
save("fig_confusion.png")

## 8. Uncertainty: bootstrap confidence intervals

**What:** resample the 25,337 test transactions *with replacement* 1,000 times, recompute every metric each time, and take the middle 95% of the results as the confidence interval.

**Why bootstrap:** net savings is a sum of a few hundred very uneven fraud losses, so it has no simple textbook formula for its standard error. The bootstrap needs no distributional assumptions and works for any metric (Efron & Tibshirani, 1993). It answers the decision owner's two real questions: *how sure are we that the savings are real*, and *how sure are we that the model beats the rule?*

**The H2 test.** In every resample we compute the model's savings, the rule's savings, and the **difference** between them. The charter's rule is: H2 is supported if the 95% confidence interval for that difference excludes zero. Because both policies are scored on the *same* resampled transactions each time, this is a paired comparison.

> This loop takes about 30–90 seconds.

In [ ]:
B = 1000
boot = {"pr": [], "roc": [], "recall": [], "precision": [], "net": [], "rule_net": [], "diff": []}
n = len(y)
IDX = []                                   # resample indices, reused for the sensitivity table
for _ in range(B):
    i = rng.integers(0, n, n)
    yb, pb, lb, rb = y[i], p[i], loss_te[i], rule[i]
    if yb.sum() == 0:
        continue
    IDX.append(i)
    fb = pb >= thr
    boot["pr"].append(average_precision_score(yb, pb))
    boot["roc"].append(roc_auc_score(yb, pb))
    boot["recall"].append((fb & (yb == 1)).sum() / yb.sum())
    boot["precision"].append((fb & (yb == 1)).sum() / max(fb.sum(), 1))
    boot["net"].append(net_savings(fb, yb, lb))
    boot["rule_net"].append(net_savings(rb, yb, lb))
    boot["diff"].append(boot["net"][-1] - boot["rule_net"][-1])
R["ci"] = {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in boot.items()}
R["diff_estimate"] = R["test_policy"]["net_savings"] - R["rule"]["net_savings"]
R["share_rule_wins"] = float(np.mean(np.array(boot["diff"]) <= 0))

DOLLARS = {"net": "model net savings", "rule_net": "rule net savings", "diff": "model minus rule"}
for k, (lo, hi) in R["ci"].items():
    if k in DOLLARS:
        print(f"{DOLLARS[k]:>18}: 95% CI [${lo:,.0f}, ${hi:,.0f}]")
    else:
        print(f"{k:>18}: 95% CI [{lo:.3f}, {hi:.3f}]")


lo, hi = R["ci"]["diff"]
h2 = lo > 0
R["H2_supported"] = bool(h2)
print()
print(f"Model saves ${R['diff_estimate']:,.0f} more than the rule, 95% CI [${lo:,.0f}, ${hi:,.0f}]")
print(f"The rule came out ahead in {R['share_rule_wins']:.1%} of resamples")
print(f"H2 check: CI for the difference {'excludes' if h2 else 'includes'} zero -> {'SUPPORTED' if h2 else 'not supported'}")

### A stricter check: resampling customers instead of transactions

The bootstrap above treats every transaction as independent. But one customer makes several transactions, and those can be alike, which would make the intervals too narrow. So we repeat the H2 test resampling whole **customers** (all of a customer's test transactions come in or out together), keeping the model and threshold fixed.

In [ ]:
cust = test.customer_id.values
uc, inv = np.unique(cust, return_inverse=True)
order = np.argsort(inv, kind="stable"); starts = np.r_[0, np.cumsum(np.bincount(inv))]
groups = [order[starts[k]:starts[k + 1]] for k in range(len(uc))]
rng_c = np.random.default_rng(SEED)
c_net, c_diff = [], []
for _ in range(B):
    pick = rng_c.integers(0, len(uc), len(uc))
    i = np.concatenate([groups[k] for k in pick])
    a = net_savings(flag[i], y[i], loss_te[i]); b = net_savings(rule[i], y[i], loss_te[i])
    c_net.append(a); c_diff.append(a - b)
R["ci_customer"] = {"customers": int(len(uc)), "txns_per_customer": float(len(y) / len(uc)),
                    "net": [float(np.percentile(c_net, 2.5)), float(np.percentile(c_net, 97.5))],
                    "diff": [float(np.percentile(c_diff, 2.5)), float(np.percentile(c_diff, 97.5))],
                    "share_rule_wins": float(np.mean(np.array(c_diff) <= 0))}
cc = R["ci_customer"]
print(f"{cc['customers']:,} test customers, {cc['txns_per_customer']:.1f} transactions each on average")
print(f"Customer-level 95% CI, model net savings: [${cc['net'][0]:,.0f}, ${cc['net'][1]:,.0f}]")
print(f"Customer-level 95% CI, model minus rule:  [${cc['diff'][0]:,.0f}, ${cc['diff'][1]:,.0f}]  "
      f"-> H2 {'still supported' if cc['diff'][0] > 0 else 'NOT supported'}")

### Report Figure B5: Uncertainty in recall and precision

In [ ]:
ci_df = pd.DataFrame({"metric": ["Recall", "Precision"],
                      "est": [tp_["recall"], tp_["precision"]],
                      "lo":  [R["ci"]["recall"][0], R["ci"]["precision"][0]],
                      "hi":  [R["ci"]["recall"][1], R["ci"]["precision"][1]]})
plt.figure(figsize=(5.5, 2.4))
plt.errorbar(ci_df.est * 100, range(2),
             xerr=[(ci_df.est - ci_df.lo) * 100, (ci_df.hi - ci_df.est) * 100],
             fmt="o", color=NAVY, ecolor=NAVY, capsize=5, ms=7)
plt.yticks(range(2), ci_df.metric); plt.xlim(0, 100); plt.xlabel("Percent (95% bootstrap CI)")
save("fig_uncertainty.png")

## 9. Diagnostics: calibration, drivers, stability, sensitivity

A model can pass its headline test and still have problems. These checks look for them.

### Report Figure 5: Calibration

**What:** group test transactions into 10 bins by score and compare the average score with the actual fraud rate. A well-calibrated model sits on the diagonal.

**Why check:** if reviewers see "0.8" they'll read it as "80% likely fraud". **Result:** the scores are badly calibrated; transactions scored about 0.8 are fraud only about 10% of the time. That's an expected side-effect of `class_weight="balanced"`, which inflates scores for the rare class (Niculescu-Mizil & Caruana, 2005). It does **not** hurt the ranking or the threshold, which we chose empirically, but scores must be recalibrated before being shown to people as probabilities.

In [ ]:
frac, mean = calibration_curve(y, p, n_bins=10, strategy="quantile")
plt.figure(figsize=(4.5, 4.0))
plt.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1, label="Perfect calibration")
plt.plot(mean, frac, color=NAVY, marker="o", label=best_name)
plt.xlabel("Predicted fraud probability"); plt.ylabel("Observed fraud rate")
plt.legend(frameon=False, fontsize=8)
save("fig_calibration.png")
R["calibration_max_gap"] = float(np.max(np.abs(frac - mean)))
print(f"Largest gap between predicted and observed: {R['calibration_max_gap']:.2f}")

### Report Figure 6: Permutation importance

**What:** shuffle one feature at a time on a 6,000-row test sample and measure how much PR-AUC drops. A big drop means the model relies on that feature.

**Why this method:** it works for any model, it's measured on held-out data, and it uses the same metric we care about. Raw logistic coefficients would be harder to compare because one-hot encoding splits each category into many columns. **Why check at all:** if the top drivers were nonsensical, the model might be exploiting a data artifact. Here they match the EDA (high-risk country, recent card velocity, international, night), which builds trust.

> Takes about 30–60 seconds.

In [ ]:
sub = test.sample(6000, random_state=SEED)
pi = permutation_importance(best, sub[NUM + CAT], sub.is_fraud, scoring="average_precision",
                            n_repeats=5, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=NUM + CAT).sort_values().tail(10)
plt.figure(figsize=(6.5, 3.6))
plt.barh(imp.index, imp.values, color=NAVY)
plt.xlabel("Drop in PR-AUC when feature is shuffled")
save("fig_importance.png")
R["importance"] = {k: float(v) for k, v in imp.sort_values(ascending=False).items()}
imp.sort_values(ascending=False).round(4)

### Stability by month

**Why:** fraud patterns drift. If performance fell sharply from October to December, the model would need frequent retraining. Reasonably flat numbers support a one-quarter pilot with monthly retraining.

In [ ]:
R["by_month"] = {}
for mth, label in [(10, "October"), (11, "November"), (12, "December")]:
    k = test.month.values == mth
    R["by_month"][label] = {"PR-AUC": float(average_precision_score(y[k], p[k])),
                            "Recall": float(((p[k] >= thr) & (y[k] == 1)).sum() / y[k].sum())}
pd.DataFrame(R["by_month"]).T.round(3)

### Report Table 3: Sensitivity to review cost and review success

**Why:** the \$5 review cost and the 80% review success rate are our two largest assumptions. As the charter promises, we re-run the whole policy for every combination of review cost (\$2, \$5, \$10, \$20) and review success (60%, 80%, 100%): for each one we **re-select the threshold on September**, apply it to the test set, compare it with the rule under the same assumptions, and compute the 95% CI for the difference using the same bootstrap resamples as Section 8.

**How to read it:** cheaper reviews justify a lower threshold and more flags; expensive reviews, or reviews that prevent less fraud, push the threshold up. The question that matters for H2 is whether the difference CI stays above zero across the table.

In [ ]:
rows = []
for sr in SENS_SUCCESS:
    for c in SENS_COSTS:
        sv_c = [savings(yva.values, pva, valid.fraud_amount_lost.values, t, c, sr) for t in grid]
        tc = float(grid[int(np.argmax(sv_c))])
        fl = p >= tc
        diffs = [net_savings(fl[i], y[i], loss_te[i], c, sr) - net_savings(rule[i], y[i], loss_te[i], c, sr) for i in IDX]
        rows.append({"Review success": f"{sr:.0%}", "Cost per review": f"${c}",
                     "base": (sr == REVIEW_SUCCESS and c == REVIEW_COST),
                     "Best threshold": tc, "Share flagged": float(fl.mean()),
                     "Model net savings": net_savings(fl, y, loss_te, c, sr),
                     "Rule net savings": net_savings(rule, y, loss_te, c, sr),
                     "Diff CI low": float(np.percentile(diffs, 2.5)),
                     "Diff CI high": float(np.percentile(diffs, 97.5))})
sens = pd.DataFrame(rows)
R["sensitivity"] = sens.to_dict(orient="records")

t3 = sens.copy()
t3["Cost per review"] = np.where(t3["base"], t3["Cost per review"] + " (base case)", t3["Cost per review"])
t3["Share flagged"] = t3["Share flagged"].map("{:.1%}".format)
usd = lambda v: f"-${-v:,.0f}" if v < 0 else f"${v:,.0f}"
for col in ["Model net savings", "Rule net savings"]:
    t3[col] = t3[col].map(usd)
t3["95% CI, model minus rule"] = [f"[{usd(a)}, {usd(b)}]" for a, b in zip(sens["Diff CI low"], sens["Diff CI high"])]
print("TABLE 3: Sensitivity to review cost and review success (threshold re-selected on September each time)")
print("Difference CI above zero in", int((sens["Diff CI low"] > 0).sum()), "of", len(sens), "scenarios")
t3.drop(columns=["base", "Diff CI low", "Diff CI high"]).set_index(["Review success", "Cost per review"])

### Robustness: do the data-quality problems change the answer?

Section 1b found fields that disagree with each other. The tempting fix is to filter those rows out, but we keep every record, for four reasons:
- **Dropping deletes a whole channel.** Every ATM row carries the `card_present` error, so filtering would remove all ATM transactions, which the policy still has to score in practice.
- **Dropping throws away fraud.** The affected rows are *more* fraud-prone than average, so removing them would make fraud look rarer than it is.
- **A live system cannot filter its inputs.** The review policy has to decide on every incoming transaction, messy ones included.
- **The target and the loss amounts are fine.** Only some descriptive fields are inconsistent. The usual rule is to fix a field when the correct value is known and otherwise keep and document it; rows are removed only when the record itself is invalid (duplicates, impossible values, out of scope).

To show this choice doesn't drive the results, we re-run the selected model three ways, re-choosing the threshold on September each time with the base-case assumptions:
1. **Keep as is:** the analysis above.
2. **Fix the fields:** set ATM rows to `card_present = 1` and raise any 24-hour spending total that is below the 1-hour total. (Day of week and weekend are no longer model inputs, so they need no fix.)
3. **Drop the inconsistent rows:** remove every row with any of those problems, plus frauds whose `fraud_type` disagrees with `card_present`.

In [ ]:
def fit_and_score(d):
    """Retrain the selected model on d, choose the threshold on September, score Oct-Dec."""
    tr, va, te = d[d.month <= 8], d[d.month == 9], d[d.month >= 10]
    m = clone(best).fit(tr[NUM + CAT], tr.is_fraud)
    pv = m.predict_proba(va[NUM + CAT])[:, 1]
    pt = m.predict_proba(te[NUM + CAT])[:, 1]
    t = float(grid[int(np.argmax([savings(va.is_fraud.values, pv, va.fraud_amount_lost.values, g) for g in grid]))])
    yt, lt, rt = te.is_fraud.values, te.fraud_amount_lost.values, te.high_risk_country.values == 1
    return {"Rows": len(d), "Frauds": int(d.is_fraud.sum()),
            "Test PR-AUC": average_precision_score(yt, pt), "Threshold": t,
            "Model net savings": savings(yt, pt, lt, t), "Rule net savings": net_savings(rt, yt, lt)}

# 2. Fix the fields where the correct value is known
fixed = df.copy()
fixed.loc[fixed.transaction_channel == "ATM", "card_present"] = 1
fixed["amount_last_24h"] = np.maximum(fixed.amount_last_24h, fixed.amount_last_1h)

# 3. Drop every row with an inconsistency
bad = ((df.amount_last_1h > df.amount_last_24h + 0.01)
       | ((df.used_pin == 1) & (df.card_present == 0))
       | ((df.fraud_type == "Card-Not-Present") & (df.card_present == 1))
       | ((df.fraud_type == "Card-Present") & (df.card_present == 0)))

rob = pd.DataFrame({"Keep as is (main analysis)": fit_and_score(df),
                    "Fix the fields": fit_and_score(fixed),
                    "Drop inconsistent rows": fit_and_score(df[~bad])}).T
R["robustness"] = rob.to_dict(orient="index")
R["robustness_dropped"] = {"rows": int(bad.sum()), "frauds": int(df.is_fraud[bad].sum()),
                           "atm_rows": int((bad & (df.transaction_channel == "ATM")).sum()),
                           "fraud_rate_dropped": float(df.is_fraud[bad].mean()),
                           "fraud_rate_kept": float(df.is_fraud[~bad].mean())}
d_ = R["robustness_dropped"]
print(f"Dropping would remove {d_['rows']:,} rows (all {d_['atm_rows']:,} ATM transactions) and {d_['frauds']} frauds; "
      f"fraud rate {d_['fraud_rate_dropped']:.2%} in dropped rows vs {d_['fraud_rate_kept']:.2%} in kept rows")
pr_gap = (rob["Test PR-AUC"] - rob.loc["Keep as is (main analysis)", "Test PR-AUC"]).abs().max()
print(f"Largest change in test PR-AUC: {pr_gap:.3f};  model beats rule in every version: "
      f"{bool((rob['Model net savings'] > rob['Rule net savings']).all())}")

show = rob.copy()
show["Rows"] = show["Rows"].astype(int).map("{:,}".format)
show["Frauds"] = show["Frauds"].astype(int)
show["Test PR-AUC"] = show["Test PR-AUC"].map("{:.3f}".format)
show["Threshold"] = show["Threshold"].map("{:.2f}".format)
for c in ["Model net savings", "Rule net savings"]:
    show[c] = show[c].map("${:,.0f}".format)
show

**Result:** all three versions agree. Test PR-AUC moves by at most a few thousandths, the threshold stays at about 0.8, and the model saves roughly twice what the rule does in every version. Dropping rows would delete every ATM transaction and about 16% of all fraud, while changing nothing about the conclusion. So the analysis keeps every record, and the report states this check in one sentence.

## 10. Fairness audit

**What:** gender and age were **not** model inputs, but a model can still treat groups differently through correlated features such as email domain or device (Barocas et al., 2023). We compare, by group:
- **False-flag rate:** share of *legitimate* customers whose transactions were held (the burden).
- **Fraud caught (recall):** share of fraud against that group that was flagged (the protection).

**Result:** false-flag rates are similar (about 4.5–5.6%), but fraud capture for customers aged 60+ is lower (about 26% vs. 35–45%). That group has only around 66 test fraud cases, so the gap is uncertain, but it warrants monitoring.

In [ ]:
tt = test.assign(flag=flag,
                 age_band=pd.cut(test.customer_age, [17, 29, 44, 59, 120],
                                 labels=["18-29", "30-44", "45-59", "60+"]))

def group_stats(d):
    return pd.Series({"Transactions": len(d),
                      "Fraud rate": d.is_fraud.mean(),
                      "False-flag rate": d[d.is_fraud == 0].flag.mean(),
                      "Fraud caught": d[d.is_fraud == 1].flag.mean(),
                      "Fraud cases": int(d.is_fraud.sum())})

R["fairness"] = {}
parts = []
for col in ["customer_gender", "age_band"]:
    g = pd.DataFrame({k: group_stats(d) for k, d in tt.groupby(col, observed=True)}).T
    R["fairness"][col] = g.round(4).to_dict(orient="index")
    parts.append(g)

t4 = pd.concat(parts)
t4.index = [f"Age {i}" if i[0].isdigit() else i for i in t4.index]
fmt = t4.copy()
fmt["Transactions"] = fmt["Transactions"].astype(int).map("{:,}".format)
fmt["Fraud cases"]  = fmt["Fraud cases"].astype(int)
for c in ["Fraud rate", "False-flag rate", "Fraud caught"]:
    fmt[c] = fmt[c].map("{:.1%}".format)
print("TABLE 4: Policy outcomes by customer group (test set)")
fmt

## Supplemental instructor-feedback checks: time denominators, fraud losses, and calendar fields

These analyses use the full synthetic 2026 file for **descriptive reporting only**. They do not change the model inputs, the January–August training split, September validation, or October–December test. The provided `transaction_day_of_week` field does not agree with the transaction timestamps; we derive weekdays from dates for this chart only. Monthly differences from a single synthetic year are not evidence of recurring seasonality.

In [ ]:
# Supplemental descriptive analysis for instructor feedback.
# The original Jan-Aug / Sep / Oct-Dec model split is NOT changed.
# All 2026 observations are used here for reporting only, after the model evaluation.
# The source transaction_day_of_week is inconsistent; weekdays are computed from timestamps.
time_view = df.copy()
time_view['derived_hour'] = time_view['transaction_datetime'].dt.hour
time_view['derived_weekday'] = time_view['transaction_datetime'].dt.day_name()
time_view['derived_month'] = time_view['transaction_datetime'].dt.month
hourly = time_view.groupby('derived_hour').agg(
    transactions=('is_fraud', 'size'),
    fraud_cases=('is_fraud', 'sum'),
    fraud_loss_usd=('fraud_amount_lost', 'sum')
).reindex(range(24), fill_value=0)
hourly['fraud_rate_pct'] = hourly['fraud_cases'].div(hourly['transactions']).mul(100)

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
                 'Friday', 'Saturday', 'Sunday']
weekday = time_view.groupby('derived_weekday').agg(
    transactions=('is_fraud', 'size'),
    fraud_cases=('is_fraud', 'sum'),
    fraud_loss_usd=('fraud_amount_lost', 'sum')
).reindex(weekday_order)
weekday['fraud_rate_pct'] = weekday['fraud_cases'].div(weekday['transactions']).mul(100)

monthly = time_view.groupby('derived_month').agg(
    transactions=('is_fraud', 'size'),
    fraud_cases=('is_fraud', 'sum'),
    fraud_loss_usd=('fraud_amount_lost', 'sum')
).reindex(range(1, 13))
monthly['fraud_rate_pct'] = monthly['fraud_cases'].div(monthly['transactions']).mul(100)

assert int(hourly['transactions'].sum()) == len(df)
assert int(hourly['fraud_cases'].sum()) == int(df['is_fraud'].sum())
assert np.isclose(hourly['fraud_loss_usd'].sum(), df['fraud_amount_lost'].sum())
assert int(weekday['transactions'].sum()) == len(df)
R['time_feedback'] = {
    'hourly': hourly.round(4).to_dict(orient='index'),
    'weekday_derived': weekday.round(4).to_dict(orient='index'),
    'monthly_descriptive_only': monthly.round(4).to_dict(orient='index'),
    'weekday_label_mismatches': int((time_view['transaction_day_of_week'] !=
                                    time_view['derived_weekday']).sum())
}
print('Supplementary time analysis: hour 03 =', hourly.loc[3].round(2).to_dict())
print('Largest total fraud loss is at hour', int(hourly['fraud_loss_usd'].idxmax()))

# Appendix Figure B10: fraud DOLLARS by hour (different from the fraud RATE).
plt.figure(figsize=(6.5, 3.3))
plt.bar(hourly.index, hourly['fraud_loss_usd'] / 1000, color='#536579', width=0.8)
plt.xlabel('Hour of transaction (0-23)')
plt.ylabel('Total fraud losses (USD thousands)')
plt.xticks(range(0, 24, 2))
save('fig_hour_loss.png')

# Appendix Figure B11: corrected weekday rates, derived from dates.
plt.figure(figsize=(6.5, 3.3))
plt.bar(range(7), weekday['fraud_rate_pct'], color='#536579', width=0.65)
plt.axhline(time_view['is_fraud'].mean() * 100, color='#888888',
            linestyle='--', linewidth=1)
plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.xlabel('Weekday derived from transaction timestamp')
plt.ylabel('Fraud rate (%)')
plt.ylim(0, 3.0)
save('fig_weekday_derived.png')

# Appendix Figure B12: a single simulated year, NOT evidence of recurring seasonality.
plt.figure(figsize=(6.5, 3.3))
plt.plot(monthly.index, monthly['fraud_rate_pct'], marker='o',
         linewidth=1.8, markersize=4, color='#536579')
plt.axhline(time_view['is_fraud'].mean() * 100, color='#888888',
            linestyle='--', linewidth=1)
plt.xticks(range(1, 13))
plt.xlabel('Month in simulated 2026 data')
plt.ylabel('Fraud rate (%)')
plt.ylim(1.5, 2.7)
save('fig_month_rate_supplement.png')


## 11. Save everything and download

Writes `results.json` (every number above), zips the `figures/` folder, and downloads both.

In [ ]:
with open("results.json", "w") as f:
    json.dump(R, f, indent=2, default=float)

import shutil
shutil.make_archive("figures", "zip", "figures")
print("Figures saved:", sorted(x.name for x in FIG.glob("*.png")))

files.download("figures.zip")
files.download("results.json")

### Figure map: file → where it appears

| File | Group Report | Charter |
|---|---|---|
| fig_risk.png | Figure 1 | |
| fig_hour.png | Figure 2 | |
| fig_threshold.png | Figure 3 | |
| fig_pr_curve.png | Figure 4 | |
| fig_calibration.png | Figure 5 | |
| fig_importance.png | Figure 6 | |
| fig_channel.png | Figure B1 | |
| fig_category.png | Figure B2 | |
| fig_amount.png | Figure B3 | |
| fig_fraudtype.png | Figure B4 | Figure 1 |
| fig_uncertainty.png | Figure B5 | |
| fig_confusion.png | Figure B6 | |
| qc_target.png, qc_distributions.png, qc_correlation.png | Not in the report yet (data-quality audit, Section 1b); good candidates for an appendix | |